# 07 · As a Tool — 02 a pipeline as a tool

**Runs end to end with no API key, and makes no network call of any kind.** The embedding path is the deterministic hash embedding from `03-embed/01-offline-embeddings.ipynb`; the corpus is three short documents written for this notebook.

Stages `01-extract` through `06-bench` are six folders, dozens of cells, and a lot of hard-won detail. An agent sees none of it. It sees this:

```json
{"name": "search_documents", "description": "...", "parameters": {"query": "string", "k": "integer"}}
```

One name, one sentence, two arguments. That collapse is the point of this notebook, and it is not a metaphor: the cells below run a real (small) extract → chunk → embed → retrieve → gate sequence, and then show it disappearing behind a single function call with a real schema. The sixth stage, `06-bench`, does not run inside the tool — it measures the tool from the outside, which is the only way you learn whether the one line is worth calling at all.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `extract_text` / `chunk_text` | Stages `01` and `02` in miniature: a raw document to normalized text to bounded chunks. | `chunk_text(extract_text(raw))` -> 2 chunks for one document |
| `hash_embed` / `build_index` | Stage `03` in miniature: the deterministic offline embedding, over every chunk. | `build_index(RAW_DOCS)` -> 5 indexed chunks, 256 dims |
| `retrieve` / `gate` | Stages `04` and `05` in miniature: cosine ranking, then a floor below which nothing is returned. | `gate(retrieve(qvec, k=5))` |
| `search_documents` | All five of the above, behind one function with one schema. | `search_documents("how does cosine similarity rank passages?", k=3)` -> the passages that cleared the gate |
| `STAGE_TRACE` | Records which internal stages ran, so one tool call visibly costs five. | one call -> `['extract', 'chunk', 'embed', 'retrieve', 'gate']` |
| The `hit@1` loop | Stage `06` in miniature: three questions with a known right source, scored against the tool. | `QUESTIONS` -> `hit@1 = 2/3` |

## Step 1 — bootstrap the repo path

Same walk-up as every other notebook in this repo: find `nbio.py` above the kernel's working directory and put the repo root on `sys.path`.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 2 — the environment, and the six stages, read off disk

No key is needed for anything here, and the readout says so. The second half of this cell lists the stage folders that actually exist next to this one, rather than asserting in prose that there are six of them.

In [ ]:
repo_root = nbio.bootstrap()
nbio.show_environment()

tools_dir = repo_root / "01-modules" / "01-tools"
stage_names = sorted(p.name for p in tools_dir.iterdir() if p.is_dir() and p.name[0].isdigit())
print()
print("stages in 01-modules/01-tools/:")
for name in stage_names:
    print(f"  {name}")

assert "07-as-a-tool" in stage_names, "this notebook's own stage should be on disk"
prior = [n for n in stage_names if n < "07-as-a-tool"]
assert len(prior) == 6, f"expected six stages before this one, found {prior}"
print()
print(f"{len(prior)} stages build the retrieval tool; this one makes it callable.")

## Step 3 — the corpus (stage `01-extract`, in miniature)

Three short documents, written for this notebook — no external dataset, no partner content. `extract_text` does the one thing extraction always does: turn a source document into plain, normalized text. The real stage does this against scanned PDFs, tables, and rotated pages; here the input is already text, and the job is reduced to dropping markdown headers and collapsing whitespace.

In [ ]:
import re

RAW_DOCS = {
    "vector-search.md": """
# Vector search

A vector embedding maps a passage of text to a point in a high-dimensional space,
so that semantic similarity becomes geometric distance.

Cosine similarity compares the angle between two such vectors rather than their
magnitude, which keeps long and short passages comparable to each other.

An index manifest records the model, the dimension, and the tokenizer used to
build a store, so a mismatch is caught before it becomes a silent empty result.
""",
    "chunking.md": """
# Chunking

A chunk is the unit that gets embedded and returned. Too large and the vector
averages several topics together; too small and it loses the context that made
it meaningful.

Token-aware chunking splits on the tokenizer's boundaries rather than on
characters, so a chunk never exceeds the embedding model's input limit.

A table split across a chunk boundary is the classic silent corruption: half the
rows land in one chunk and the header lands in another.
""",
    "grounding.md": """
# Grounding

A grounding check compares a generated answer against the context it was
supposed to come from, and refuses the answer when a quoted span is not present
in that context.

Abstention is a feature, not a failure: returning nothing is better than
returning a confident answer with no support behind it.
""",
}


def extract_text(raw: str) -> str:
    """Normalize a source document to plain text: drop headers, collapse blank runs."""
    kept = [ln.strip() for ln in raw.splitlines() if not ln.strip().startswith("#")]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(kept)).strip()


sample = extract_text(RAW_DOCS["vector-search.md"])
print(f"{len(RAW_DOCS)} documents")
print(f"vector-search.md: {len(RAW_DOCS['vector-search.md'])} raw chars -> {len(sample)} extracted chars")
print()
print(sample[:180] + " ...")

## Step 4 — chunking (stage `02-chunk`, in miniature)

Paragraphs, merged up to a character budget. The real stage counts tokens with the embedding model's own tokenizer and keeps tables intact; this one counts characters, which is the honest simplification a notebook can make while still producing chunks of the right shape.

In [ ]:
def chunk_text(text: str, max_chars: int = 320) -> list[str]:
    """Merge paragraphs into chunks no longer than max_chars."""
    chunks: list[str] = []
    current = ""
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        para = " ".join(para.split())
        if current and len(current) + len(para) + 1 > max_chars:
            chunks.append(current)
            current = para
        else:
            current = f"{current} {para}".strip()
    if current:
        chunks.append(current)
    return chunks


demo_chunks = chunk_text(sample)
nbio.table(
    [(i, len(c), c[:58] + "...") for i, c in enumerate(demo_chunks)],
    headers=("#", "chars", "chunk"),
)
assert all(len(c) <= 320 for c in demo_chunks), "a chunk exceeded the character budget"

## Step 5 — embedding and the index (stage `03-embed`, in miniature)

`hash_embed` is the same deterministic, offline embedding `03-embed/01-offline-embeddings.ipynb` defines: SHA-256 over whitespace tokens, accumulated into a fixed-size vector, L2-normalized. Real wiring, fake semantics — it will rank a chunk that shares vocabulary with the query, not a chunk that shares meaning. Everything downstream is built on that limitation, and the numbers below should be read with it in mind.

In [ ]:
import hashlib
import math

DIM = 256


def hash_embed(text: str, dim: int = DIM) -> list[float]:
    """Deterministic, offline embedding -- no model, no API key, no network."""
    vec = [0.0] * dim
    for tok in (text or "").lower().split():
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        vec[h % dim] += 1.0 if (h >> 8) & 1 else -1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def build_index(docs: dict[str, str]) -> list[dict]:
    """Run extract -> chunk -> embed over every document, once."""
    index: list[dict] = []
    for source, raw in sorted(docs.items()):
        for i, chunk in enumerate(chunk_text(extract_text(raw))):
            index.append(
                {
                    "chunk_id": f"{source}#{i}",
                    "source": source,
                    "text": chunk,
                    "embedding": hash_embed(chunk),
                }
            )
    return index


INDEX = build_index(RAW_DOCS)
print(f"{len(INDEX)} chunks indexed from {len(RAW_DOCS)} documents, dim={len(INDEX[0]['embedding'])}")
nbio.table(
    [(c["chunk_id"], len(c["text"])) for c in INDEX],
    headers=("chunk_id", "chars"),
)

## Step 6 — retrieve and gate (stages `04-retrieve` and `05-gate`, in miniature)

`retrieve` cosine-ranks the index against a query vector. `gate` applies a floor: below it, the tool returns nothing rather than returning its least-bad guess. The floor is a neutral illustrative value, not a tuned or product setting — what matters is that it exists, and that "no result" is a result the caller has to handle.

In [ ]:
SCORE_FLOOR = 0.08  # illustrative, not a tuned value


def _cosine(a: list[float], b: list[float]) -> float:
    if not a or not b or len(a) != len(b):
        return 0.0
    return sum(x * y for x, y in zip(a, b))  # both vectors are already unit length


def retrieve(qvec: list[float], k: int) -> list[dict]:
    scored = [{**c, "score": _cosine(qvec, c["embedding"])} for c in INDEX]
    scored.sort(key=lambda c: -c["score"])
    return scored[:k]


def gate(hits: list[dict], floor: float = SCORE_FLOOR) -> list[dict]:
    """Drop anything below the floor. An empty list is a legitimate answer."""
    return [h for h in hits if h["score"] >= floor]

## Step 7 — the whole pipeline, as one function with one schema

This is the keystone cell of the stage. `search_documents` runs all five stages and returns a plain list of dicts. Its signature is the contract an agent gets — `query` required, `k` defaulted — and its docstring is the only description of it the model will ever read (the lesson of `01-function-as-tool.ipynb`, applied to something that actually does work).

`STAGE_TRACE` is not part of the tool's behaviour; it exists so the next cell can show what one call costs internally.

In [ ]:
import inspect
import json
import typing

STAGE_TRACE: list[str] = []


def search_documents(query: str, k: int = 5) -> list[dict]:
    """Search the indexed document corpus and return the passages most relevant to a query.

    Use this for any question whose answer would be found in the documents: vector
    search, chunking, and grounding. Returns an empty list when nothing in the corpus
    is relevant enough to trust.

    Args:
        query: The question or topic to search for, in natural language.
        k: How many passages to return, most relevant first.
    """
    STAGE_TRACE.clear()
    STAGE_TRACE.extend(["extract", "chunk", "embed"])  # done once, at index build time
    qvec = hash_embed(query)
    hits = retrieve(qvec, k)
    STAGE_TRACE.append("retrieve")
    kept = gate(hits)
    STAGE_TRACE.append("gate")
    return [
        {"chunk_id": h["chunk_id"], "source": h["source"], "text": h["text"], "score": round(h["score"], 4)}
        for h in kept
    ]

## Step 8 — the spec the agent receives

Built the same way `01-function-as-tool.ipynb` builds it, inlined here so this notebook stands on its own. Read what is *not* in it: no mention of chunk size, no embedding dimension, no score floor, no index. Six stages of design decisions, and the agent's entire view of them is a sentence and two parameters.

In [ ]:
_JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool_spec(fn) -> dict:
    """The name, description and argument schema a model is shown for one function."""
    doc = inspect.getdoc(fn) or ""
    head, arg_docs, in_args = [], {}, False
    for line in doc.splitlines():
        stripped = line.strip()
        if stripped.lower().rstrip(":") in {"args", "arguments", "parameters"}:
            in_args = True
            continue
        if in_args:
            if stripped and ":" in stripped:
                name, _, desc = stripped.partition(":")
                arg_docs[name.strip()] = desc.strip()
        else:
            head.append(stripped)

    hints = typing.get_type_hints(fn)
    properties, required = {}, []
    for name, param in inspect.signature(fn).parameters.items():
        prop = {"type": _JSON_TYPES.get(hints.get(name, str), "string")}
        if name in arg_docs:
            prop["description"] = arg_docs[name]
        if param.default is inspect.Parameter.empty:
            required.append(name)
        else:
            prop["default"] = param.default
        properties[name] = prop

    return {
        "name": fn.__name__,
        "description": " ".join(p for p in head if p).strip(),
        "parameters": {"type": "object", "properties": properties, "required": required},
    }


SEARCH_SPEC = tool_spec(search_documents)
print(json.dumps(SEARCH_SPEC, indent=2))

assert SEARCH_SPEC["parameters"]["required"] == ["query"]
assert SEARCH_SPEC["parameters"]["properties"]["k"]["default"] == 5

## Step 9 — one line, five stages

From the agent's side this is a single call. `STAGE_TRACE` shows what that single call actually put in motion.

In [ ]:
results = search_documents("how does cosine similarity rank passages?", k=3)

nbio.table(
    [(r["chunk_id"], r["score"], r["text"][:56] + "...") for r in results],
    headers=("chunk_id", "score", "text"),
)
print()
print(f"one call from the agent's side : search_documents(query, k=3)")
print(f"stages that ran inside it      : {STAGE_TRACE}")

assert len(results) <= 3, "k must cap the number of results"
assert STAGE_TRACE == ["extract", "chunk", "embed", "retrieve", "gate"]
assert set(results[0]) == {"chunk_id", "source", "text", "score"}, "the returned shape is the contract"

## Step 10 — the six stages, named against what runs here

Each row is a real folder in `01-modules/01-tools/`. The right-hand column is the honest part: what the wrapper above leaves out.

In [ ]:
MAPPING = [
    ("01-extract", "extract_text", "OCR, tables, page orientation, handwriting"),
    ("02-chunk", "chunk_text", "token-aware splitting, tables kept intact, store backends"),
    ("03-embed", "hash_embed", "a real embedding model, 3072 dims, a persistent vector store"),
    ("04-retrieve", "retrieve", "six-source fanout, dedupe/merge, LLM chunk scoring, five-signal ranking"),
    ("05-gate", "gate", "quote-level grounding against a generated answer, abstention"),
    ("06-bench", "the hit@1 loop (Step 13)", "20 questions, DeepEval metrics, a gold-context arm, a leaderboard"),
]
nbio.table(MAPPING, headers=("stage", "here", "what this notebook leaves out"))

on_disk = {name for name in stage_names}
for stage, _, _ in MAPPING:
    assert stage in on_disk, f"{stage} is named here but not on disk"
print()
print("every stage named above exists on disk next to this notebook")

## Step 11 — the answer the tool refuses to give, and the one it should have refused

Two off-topic queries, and they do not behave the same way. The first is refused: every candidate scores below the floor, `search_documents` returns an empty list, and that empty list is the correct output — the failure it prevents is the tool handing back its least-bad chunk and the model reading it as evidence.

The second is not refused, and it is the more useful of the two to look at. It is just as off-topic, it scores three times higher, and it sails through the gate — because the hash embedding has no semantics, so shared *function words* ("in", "the") are worth exactly as much as shared subject-matter words. A score floor over meaningless vectors is not a relevance guarantee; it is a floor on a number. The next cell prints the words that actually earned that score.

In [ ]:
refused_q = "Trombone repertoire Baroque Telemann sonatas"
slipped_q = "quarterly revenue in the Latin American region"

refused = search_documents(refused_q, k=3)
refused_raw = retrieve(hash_embed(refused_q), k=3)
slipped = search_documents(slipped_q, k=3)

nbio.table(
    [(h["chunk_id"], round(h["score"], 4)) for h in refused_raw],
    headers=(f"ranked for {refused_q!r}", "score (floor is 0.08)"),
)
print(f"search_documents({refused_q!r}) -> {refused}")
print()
nbio.table(
    [(h["chunk_id"], h["score"]) for h in slipped],
    headers=(f"returned for {slipped_q!r}", "score"),
)

assert refused == [], "every candidate was below the floor, so nothing should be returned"
assert refused_raw, "the ranker still had candidates -- the gate is what refused them"
assert slipped, "this off-topic query is expected to clear the floor, which is the point being made"

## Step 12 — what earned that score

The overlap between the second query and the chunk that beat the floor, printed rather than described. Nothing on this list is about the query's subject.

In [ ]:
top_chunk = next(c for c in INDEX if c["chunk_id"] == slipped[0]["chunk_id"])
shared = sorted(set(slipped_q.lower().split()) & set(top_chunk["text"].lower().split()))

print(f"query : {slipped_q}")
print(f"chunk : {top_chunk['text'][:96]}...")
print(f"score : {slipped[0]['score']}")
print(f"words shared by both : {shared}")

assert shared, "the score has to have come from some shared token"
assert not ({"quarterly", "revenue", "latin", "american", "region"} & set(shared)), \
    "none of the query's subject-matter words appear in the chunk"
print()
print("The gate did its job as specified and still let an irrelevant passage through.")
print("A floor on a score is not a check on meaning -- that is what 05-gate's grounding")
print("check, which compares an answer against the text it cites, exists to do instead.")

## Step 13 — stage `06-bench`, from outside the tool

The five stages above run *inside* every call. Benchmarking runs outside it: three questions whose correct source document is known in advance, scored on whether the top hit came from the right document. Three questions is far too few to conclude anything — it is here to show the shape, and to make one honest point immediately below the number.

In [ ]:
QUESTIONS = [
    ("what does a vector embedding map text to?", "vector-search.md"),
    ("why is a chunk that is too large a problem?", "chunking.md"),
    ("when should a system refuse to answer?", "grounding.md"),
]

rows, hits = [], 0
for question, expected in QUESTIONS:
    got = search_documents(question, k=1)
    top = got[0]["source"] if got else "(nothing returned)"
    ok = top == expected
    hits += ok
    rows.append((question[:46], expected, top, "hit" if ok else "miss"))

nbio.table(rows, headers=("question", "expected source", "top source", ""))
print()
print(f"hit@1 = {hits}/{len(QUESTIONS)} on a 3-question set with hash embeddings")
print("A number this small measures the harness, not the retriever. Stage 06-bench is where")
print("the real version of this lives, including a documented, still-open flakiness.")

assert 0 <= hits <= len(QUESTIONS)

## What did not come across

- **The retrieval this wraps is a miniature.** No OCR, no token-aware chunking, no real embedding model, no persistent vector store, no fanout across six sources, no dedupe or merge, no LLM relevance scoring, no retraction check, no rate limiter, no five-signal ranking blend, no multi-query. Every one of those is a real, running notebook in stages `01`–`06`; none of them is in `search_documents`. What transfers from this notebook is the *shape* — a schema, a docstring, a return contract, a gate — not the retrieval quality.
- **Hash embeddings have no semantics.** Ranking here rewards shared vocabulary. A question phrased entirely in synonyms retrieves nothing useful, and the `hit@1` in Step 12 would move under a real embedding model.
- **The index is built once, in the notebook, and lives in memory.** A real tool call does not re-run extract and chunk; this notebook's `STAGE_TRACE` lists them for every call because those stages are what *produced* the index, not because they run per query. Freshness, incremental indexing, and invalidation are all absent.
- **No argument validation.** `search_documents("", k=-4)` is not rejected here. What a tool should do with arguments it cannot honour is `04-tool-failure.ipynb`.

Next: `03-a-second-tool.ipynb` — a second tool with no retrieval in it at all, and the choice that appears the moment it exists.